# 03 - Vegetation indices within the tower footprint polygon

For every HLS scene, this notebook clips the bands to a fixed footprint polygon around the tower
(`footprint_160m/<SITE>_footprint.shp`), applies the Fmask cloud mask, computes seven spectral indices
and summarises each one as the mean and the median over the polygon.

The output tables (one per sensor and site) are merged in notebook 05. Run this notebook four
times: Landsat and Sentinel for each of the two sites.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio import mask

In [ ]:
SITE = "BAU"          # "ASP" or "BAU"
SENSOR = "Landsat"    # "Landsat" or "Sentinel"

HLS_DIR = Path("../data/raw/hls") / SENSOR
FOOTPRINT_SHP = Path(f"../data/raw/shapefiles/footprint_160m/{SITE}_footprint.shp")
OUT_DIR = Path("../data/processed/vi_tables/fixed_footprint")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BANDS = {
    "Landsat":  {"blue": "B02", "red": "B04", "nir": "B05", "swir1": "B06"},
    "Sentinel": {"blue": "B02", "red": "B04", "nir": "B8A", "swir1": "B11"},
}[SENSOR]

footprint = gpd.read_file(FOOTPRINT_SHP)

## Cloud mask and indices

Fmask bits 1 (cloud), 3 (cloud shadow) and 4 (snow/ice) must all be 0 for a pixel to be used.

| index | formula |
|---|---|
| NDVI | (NIR - Red) / (NIR + Red) |
| EVI | 2.5 (NIR - Red) / (NIR + 6 Red - 7.5 Blue + 1) |
| SAVI | 1.5 (NIR - Red) / (NIR + Red + 0.5) |
| MSAVI | (2 NIR + 1 - sqrt((2 NIR + 1)^2 - 8 (NIR - Red))) / 2 |
| NDWI (Gao) | (NIR - SWIR1) / (NIR + SWIR1) |
| MSI | SWIR1 / NIR |
| SR | NIR / Red |

In [ ]:
def clear_sky_values(fmask):
    good = []
    for v in np.unique(fmask):
        v = int(v)
        cloud, shadow, snow = (v >> 1) & 1, (v >> 3) & 1, (v >> 4) & 1
        if not (cloud or shadow or snow):
            good.append(v)
    return good


def find_band(folder, key):
    matches = sorted(f for f in folder.iterdir() if key in f.name)
    if not matches:
        raise FileNotFoundError(f"{key} not found in {folder.name}")
    return matches[0]


def read_clipped(path, shapes, nodata=-9999):
    with rasterio.open(path) as src:
        geoms = shapes.to_crs(src.crs).geometry if shapes.crs != src.crs else shapes.geometry
        arr, _ = mask.mask(src, geoms, crop=True, nodata=nodata)
    return arr[0]


def spectral_indices(folder):
    paths = {k: find_band(folder, v) for k, v in BANDS.items()}
    fmask_path = find_band(folder, "Fmask")

    with rasterio.open(fmask_path) as src:
        good = clear_sky_values(src.read(1))
    fmask = read_clipped(fmask_path, footprint, nodata=255)   # 255 is the HLS Fmask fill value
    clear = np.isin(fmask, good)

    b = {}
    for k, p in paths.items():
        arr = read_clipped(p, footprint).astype("float64")
        b[k] = np.where(arr == -9999, np.nan, arr) * 0.0001

    blue, red, nir, swir1 = b["blue"], b["red"], b["nir"], b["swir1"]
    with np.errstate(divide="ignore", invalid="ignore"):
        out = {
            "NDVI": (nir - red) / (nir + red),
            "EVI": 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0),
            "SAVI": 1.5 * (nir - red) / (nir + red + 0.5),
            "MSAVI": (2 * nir + 1 - np.sqrt((2 * nir + 1) ** 2 - 8 * (nir - red))) / 2,
            "NDWI": (nir - swir1) / (nir + swir1),
            "MSI": swir1 / nir,
            "SR": nir / red,
        }
    # cloudy pixels are dropped after the index is computed
    return {k: np.where(clear, v, np.nan) for k, v in out.items()}

## Loop over scenes

Scenes with no clear pixels inside the polygon give NaN (numpy warns about an empty slice for those).

In [ ]:
records = []
for folder in sorted(HLS_DIR.iterdir()):
    if not folder.is_dir():
        continue
    try:
        idx = spectral_indices(folder)
    except FileNotFoundError as e:
        print(f"skipping {folder.name}: {e}")
        continue

    row = {"Date": folder.name[1:]}              # folder names are L/S + YYYY-MM-DD
    for name, arr in idx.items():
        row[f"{name}_mean"] = np.nanmean(arr)
        row[f"{name}_median"] = np.nanmedian(arr)
    records.append(row)

vi_table = pd.DataFrame(records)
# same column order as before: all means first, then all medians
names = ["NDVI", "EVI", "SAVI", "MSAVI", "NDWI", "MSI", "SR"]
vi_table = vi_table[["Date"] + [f"{n}_mean" for n in names] + [f"{n}_median" for n in names]]

out_path = OUT_DIR / f"{SENSOR}_{SITE}.csv"
vi_table.to_csv(out_path, index=False)
print(f"{len(vi_table)} scenes -> {out_path}")
vi_table.head()